 00_exploration_vllm — inspect raw vLLM benchmark outputs

In [1]:
from pathlib import Path
import pandas as pd

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 200)

ROOT = Path.cwd().resolve().parents[0] if Path.cwd().name == "notebooks" else Path.cwd().resolve()
RESULTS_DIR = ROOT / "results" / "raw"

DATASETS = ["mmlu", "hellaswag", "gsm8k", "dolly"]

print("ROOT:", ROOT)
print("RESULTS_DIR:", RESULTS_DIR)
print("RESULTS_DIR exists:", RESULTS_DIR.exists())


ROOT: /home/stephane/resource-efficiency-benchmark
RESULTS_DIR: /home/stephane/resource-efficiency-benchmark/results/raw
RESULTS_DIR exists: True


 Discover vLLM runs under results/raw/<model_alias>/<run_id>/

In [2]:
run_rows = []

if RESULTS_DIR.exists():
    for model_dir in sorted(RESULTS_DIR.iterdir()):
        if not model_dir.is_dir():
            continue

        for run_dir in sorted(model_dir.iterdir()):
            if not run_dir.is_dir():
                continue

            run_rows.append({
                "model_alias_dir": model_dir.name,
                "run_id": run_dir.name,
                "run_dir": str(run_dir),
            })

runs_df = pd.DataFrame(run_rows)
runs_df.head()


,model_alias_dir,run_id,run_dir
0,llama3_2_1b,2026-04-07_13-35-14,/home/stephane/resource-efficiency-benchmark/r...
1,ministral3_3b_awq,2026-04-07_13-56-00,/home/stephane/resource-efficiency-benchmark/r...
2,phi4mini_gptq_4bit,2026-04-07_16-29-17,/home/stephane/resource-efficiency-benchmark/r...
3,qwen2_5_1_5b,2026-04-07_14-29-28,/home/stephane/resource-efficiency-benchmark/r...
4,qwen3_4b_awq,2026-04-07_14-50-20,/home/stephane/resource-efficiency-benchmark/r...


In [3]:
print("Number of vLLM runs discovered:", len(runs_df))
runs_df


Number of vLLM runs discovered: 5


,model_alias_dir,run_id,run_dir
0,llama3_2_1b,2026-04-07_13-35-14,/home/stephane/resource-efficiency-benchmark/r...
1,ministral3_3b_awq,2026-04-07_13-56-00,/home/stephane/resource-efficiency-benchmark/r...
2,phi4mini_gptq_4bit,2026-04-07_16-29-17,/home/stephane/resource-efficiency-benchmark/r...
3,qwen2_5_1_5b,2026-04-07_14-29-28,/home/stephane/resource-efficiency-benchmark/r...
4,qwen3_4b_awq,2026-04-07_14-50-20,/home/stephane/resource-efficiency-benchmark/r...


 Audit expected files for each vLLM run

In [4]:
audit_rows = []

for _, r in runs_df.iterrows():
    run_dir = Path(r["run_dir"])

    record = {
        "model_alias_dir": r["model_alias_dir"],
        "run_id": r["run_id"],
        "config_yaml": (run_dir / "config.yaml").exists(),
        "server_log": (run_dir / "server.log").exists(),
    }

    for ds in DATASETS:
        record[f"{ds}_summary"] = (run_dir / f"{ds}.csv").exists()
        record[f"{ds}_samples"] = (run_dir / f"{ds}_samples.csv").exists()

    audit_rows.append(record)

audit_df = pd.DataFrame(audit_rows)
audit_df.head()


,model_alias_dir,run_id,config_yaml,server_log,mmlu_summary,mmlu_samples,hellaswag_summary,hellaswag_samples,gsm8k_summary,gsm8k_samples,dolly_summary,dolly_samples
0,llama3_2_1b,2026-04-07_13-35-14,True,True,True,True,True,True,True,True,True,True
1,ministral3_3b_awq,2026-04-07_13-56-00,True,True,True,True,True,True,True,True,True,True
2,phi4mini_gptq_4bit,2026-04-07_16-29-17,True,True,True,True,True,True,True,True,True,True
3,qwen2_5_1_5b,2026-04-07_14-29-28,True,True,True,True,True,True,True,True,True,True
4,qwen3_4b_awq,2026-04-07_14-50-20,True,True,True,True,True,True,True,True,True,True


In [5]:
audit_df


,model_alias_dir,run_id,config_yaml,server_log,mmlu_summary,mmlu_samples,hellaswag_summary,hellaswag_samples,gsm8k_summary,gsm8k_samples,dolly_summary,dolly_samples
0,llama3_2_1b,2026-04-07_13-35-14,True,True,True,True,True,True,True,True,True,True
1,ministral3_3b_awq,2026-04-07_13-56-00,True,True,True,True,True,True,True,True,True,True
2,phi4mini_gptq_4bit,2026-04-07_16-29-17,True,True,True,True,True,True,True,True,True,True
3,qwen2_5_1_5b,2026-04-07_14-29-28,True,True,True,True,True,True,True,True,True,True
4,qwen3_4b_awq,2026-04-07_14-50-20,True,True,True,True,True,True,True,True,True,True


 Compact audit summary by vLLM model alias

In [6]:
if not audit_df.empty:
    audit_summary = (
        audit_df.groupby("model_alias_dir")
        .agg(
            n_runs=("run_id", "nunique"),
            config_yaml=("config_yaml", "sum"),
            server_log=("server_log", "sum"),
            mmlu_summary=("mmlu_summary", "sum"),
            mmlu_samples=("mmlu_samples", "sum"),
            hellaswag_summary=("hellaswag_summary", "sum"),
            hellaswag_samples=("hellaswag_samples", "sum"),
            gsm8k_summary=("gsm8k_summary", "sum"),
            gsm8k_samples=("gsm8k_samples", "sum"),
            dolly_summary=("dolly_summary", "sum"),
            dolly_samples=("dolly_samples", "sum"),
        )
        .reset_index()
    )
    display(audit_summary)


,model_alias_dir,n_runs,config_yaml,server_log,mmlu_summary,mmlu_samples,hellaswag_summary,hellaswag_samples,gsm8k_summary,gsm8k_samples,dolly_summary,dolly_samples
0,llama3_2_1b,1,1,1,1,1,1,1,1,1,1,1
1,ministral3_3b_awq,1,1,1,1,1,1,1,1,1,1,1
2,phi4mini_gptq_4bit,1,1,1,1,1,1,1,1,1,1,1
3,qwen2_5_1_5b,1,1,1,1,1,1,1,1,1,1,1
4,qwen3_4b_awq,1,1,1,1,1,1,1,1,1,1,1


 Find potentially incomplete vLLM runs

In [7]:
if not audit_df.empty:
    missing_any_common = ~audit_df[["config_yaml", "server_log"]].all(axis=1)

    dataset_cols = []
    for ds in DATASETS:
        dataset_cols.extend([f"{ds}_summary", f"{ds}_samples"])

    missing_all_dataset_outputs = ~audit_df[dataset_cols].any(axis=1)

    incomplete_df = audit_df[missing_any_common | missing_all_dataset_outputs].copy()
    display(incomplete_df)


,model_alias_dir,run_id,config_yaml,server_log,mmlu_summary,mmlu_samples,hellaswag_summary,hellaswag_samples,gsm8k_summary,gsm8k_samples,dolly_summary,dolly_samples


 Load all vLLM summary and sample files

In [8]:
summary_tables = []
sample_tables = []

for _, r in runs_df.iterrows():
    run_dir = Path(r["run_dir"])

    for ds in DATASETS:
        summary_path = run_dir / f"{ds}.csv"
        samples_path = run_dir / f"{ds}_samples.csv"

        if summary_path.exists():
            df = pd.read_csv(summary_path)
            df["model_alias_dir"] = r["model_alias_dir"]
            df["run_id"] = r["run_id"]
            df["dataset"] = ds
            df["source_path"] = str(summary_path)
            summary_tables.append(df)

        if samples_path.exists():
            df = pd.read_csv(samples_path)
            df["model_alias_dir"] = r["model_alias_dir"]
            df["run_id"] = r["run_id"]
            df["dataset"] = ds
            df["source_path"] = str(samples_path)
            sample_tables.append(df)

summaries = pd.concat(summary_tables, ignore_index=True) if summary_tables else pd.DataFrame()
samples = pd.concat(sample_tables, ignore_index=True) if sample_tables else pd.DataFrame()

print("summaries shape:", summaries.shape)
print("samples shape:", samples.shape)


summaries shape: (20, 34)
samples shape: (15500, 35)


 Inspect raw vLLM summary files

In [9]:
summaries.head()


,Model alias,Repo id,Served model id,Accuracy mean,Accuracy sd,TTFT ms mean,TTFT ms sd,End-to-end latency ms mean,End-to-end latency ms sd,Tokens/sec mean,Tokens/sec sd,CPU percent avg mean,CPU percent avg sd,Temp C avg mean,Temp C avg sd,GPU percent avg mean,GPU percent avg sd,Power W avg mean,Power W avg sd,Sequential sustained tokens/sec mean,Sequential sustained tokens/sec sd,Sequential sustained samples/sec mean,Sequential sustained samples/sec sd,CPU percent peak,RAM used MB peak,Process RAM MB peak,Temp C peak,GPU percent peak,VRAM used MB peak,Power W peak,model_alias_dir,run_id,dataset,source_path
0,llama3_2_1b,unsloth/Llama-3.2-1B-Instruct,unsloth/Llama-3.2-1B-Instruct,0.44,0.0,68.2622,0.3665,122.6156,0.7112,18.4056,0.1139,6.0300,0.2637,68.5707,0.6288,8.2657,9.4386,58.1622,0.6932,5.6841,0.0316,5.6841,0.0316,39.2,9376.24,896.96,71.500,22.0,5436.0,61.44,llama3_2_1b,2026-04-07_13-35-14,mmlu,/home/stephane/resource-efficiency-benchmark/r...
1,llama3_2_1b,unsloth/Llama-3.2-1B-Instruct,unsloth/Llama-3.2-1B-Instruct,0.24,0.0,68.1985,0.1777,122.4676,0.4120,18.4337,0.0793,5.9284,0.1299,67.6965,0.2944,2.5020,6.6964,58.2584,0.0683,5.6900,0.0191,5.6900,0.0191,43.4,9272.16,898.32,68.625,22.0,5400.0,60.20,llama3_2_1b,2026-04-07_13-35-14,hellaswag,/home/stephane/resource-efficiency-benchmark/r...
2,llama3_2_1b,unsloth/Llama-3.2-1B-Instruct,unsloth/Llama-3.2-1B-Instruct,0.06,0.0,67.0925,0.2472,138.3586,0.3620,53.9784,0.0768,6.8413,0.0165,71.7561,0.2473,44.0157,15.2163,103.1411,1.4601,23.6684,0.0713,5.2364,0.0158,23.6,9387.93,899.57,73.250,100.0,5370.0,128.19,llama3_2_1b,2026-04-07_13-35-14,gsm8k,/home/stephane/resource-efficiency-benchmark/r...
3,llama3_2_1b,unsloth/Llama-3.2-1B-Instruct,unsloth/Llama-3.2-1B-Instruct,NaN,NaN,73.7217,5.8875,471.5200,54.9763,109.4824,4.3976,7.9899,0.2048,74.9008,0.1878,66.0237,30.1451,158.7051,4.3896,83.8610,4.9172,1.9239,0.1934,22.2,9301.80,899.69,76.250,100.0,5158.0,187.04,llama3_2_1b,2026-04-07_13-35-14,dolly,/home/stephane/resource-efficiency-benchmark/r...
4,ministral3_3b_awq,cyankiwi/Ministral-3-3B-Instruct-2512-AWQ-4bit,cyankiwi/Ministral-3-3B-Instruct-2512-AWQ-4bit,0.64,0.0,82.6019,0.9471,137.8528,0.9156,18.1055,0.0572,5.9592,0.0430,70.0784,0.2781,28.6720,8.2209,69.6547,1.8607,5.2170,0.0256,5.2170,0.0256,23.9,10783.76,907.05,73.250,100.0,5396.0,102.96,ministral3_3b_awq,2026-04-07_13-56-00,mmlu,/home/stephane/resource-efficiency-benchmark/r...


In [10]:
list(summaries.columns)


['Model alias',
 'Repo id',
 'Served model id',
 'Accuracy mean',
 'Accuracy sd',
 'TTFT ms mean',
 'TTFT ms sd',
 'End-to-end latency ms mean',
 'End-to-end latency ms sd',
 'Tokens/sec mean',
 'Tokens/sec sd',
 'CPU percent avg mean',
 'CPU percent avg sd',
 'Temp C avg mean',
 'Temp C avg sd',
 'GPU percent avg mean',
 'GPU percent avg sd',
 'Power W avg mean',
 'Power W avg sd',
 'Sequential sustained tokens/sec mean',
 'Sequential sustained tokens/sec sd',
 'Sequential sustained samples/sec mean',
 'Sequential sustained samples/sec sd',
 'CPU percent peak',
 'RAM used MB peak',
 'Process RAM MB peak',
 'Temp C peak',
 'GPU percent peak',
 'VRAM used MB peak',
 'Power W peak',
 'model_alias_dir',
 'run_id',
 'dataset',
 'source_path']

In [11]:
if not summaries.empty:
    display(
        summaries.groupby(["model_alias_dir", "dataset"])
        .size()
        .rename("n_rows")
        .reset_index()
        .sort_values(["model_alias_dir", "dataset"])
    )


,model_alias_dir,dataset,n_rows
0,llama3_2_1b,dolly,1
1,llama3_2_1b,gsm8k,1
2,llama3_2_1b,hellaswag,1
3,llama3_2_1b,mmlu,1
4,ministral3_3b_awq,dolly,1
5,ministral3_3b_awq,gsm8k,1
6,ministral3_3b_awq,hellaswag,1
7,ministral3_3b_awq,mmlu,1
8,phi4mini_gptq_4bit,dolly,1
9,phi4mini_gptq_4bit,gsm8k,1


 Inspect raw vLLM sample files

In [12]:
samples.head()


,repo_id,served_model_id,model_alias,dataset_name,run_idx,sample_idx,prompt,question,choices,instruction,context,gold_answer,parsed_prediction,correct,raw_output,output_tokens,ttft_ms,end_to_end_latency_ms,tokens_per_sec,cpu_percent_avg,cpu_percent_peak,ram_used_mb_peak,process_ram_mb_peak,temp_c_avg,temp_c_peak,gpu_util_percent_avg,gpu_util_percent_peak,vram_used_mb_peak,power_w_avg,power_w_peak,model_alias_dir,run_id,dataset,source_path,metric_points_json
0,unsloth/Llama-3.2-1B-Instruct,unsloth/Llama-3.2-1B-Instruct,llama3_2_1b,mmlu,1,1,"In primates, the location of which of the foll...","In primates, the location of which of the foll...",the feet || the skull || the foramen magnum ||...,NaN,NaN,2.0,C,True,C,1,79.3070,133.0174,18.6184,4.6000,6.3,9370.37,895.34,68.6250,68.625,3.0,3.0,5254.0,47.9233,48.40,llama3_2_1b,2026-04-07_13-35-14,mmlu,/home/stephane/resource-efficiency-benchmark/r...,NaN
1,unsloth/Llama-3.2-1B-Instruct,unsloth/Llama-3.2-1B-Instruct,llama3_2_1b,mmlu,1,2,In the comic strip 'Peanuts' what is Schroeder...,In the comic strip 'Peanuts' what is Schroeder...,Dancing || playing football || playing the pia...,NaN,NaN,2.0,C,True,C,1,67.7943,122.0332,18.4369,6.7000,7.3,9375.46,895.34,68.6250,68.625,3.0,3.0,5254.0,49.9367,50.54,llama3_2_1b,2026-04-07_13-35-14,mmlu,/home/stephane/resource-efficiency-benchmark/r...,NaN
2,unsloth/Llama-3.2-1B-Instruct,unsloth/Llama-3.2-1B-Instruct,llama3_2_1b,mmlu,1,3,The Yang-shao culture gave way to the Lung-Sha...,The Yang-shao culture gave way to the Lung-Sha...,"6,000 B.P. || 5,000 B.P. || 4,000 B.P. || 3,00...",NaN,NaN,1.0,C,False,C,1,67.4295,122.5596,18.1389,5.6000,7.3,9376.24,895.34,68.5833,68.625,3.0,3.0,5254.0,51.2400,52.44,llama3_2_1b,2026-04-07_13-35-14,mmlu,/home/stephane/resource-efficiency-benchmark/r...,NaN
3,unsloth/Llama-3.2-1B-Instruct,unsloth/Llama-3.2-1B-Instruct,llama3_2_1b,mmlu,1,4,At which of the following locations does bile ...,At which of the following locations does bile ...,Gastroesophageal sphincter || Duodenum || Ileo...,NaN,NaN,1.0,C,False,C,1,66.5789,121.0383,18.3623,6.0000,7.3,9353.32,895.34,68.5833,68.625,3.0,3.0,5254.0,52.9467,53.94,llama3_2_1b,2026-04-07_13-35-14,mmlu,/home/stephane/resource-efficiency-benchmark/r...,NaN
4,unsloth/Llama-3.2-1B-Instruct,unsloth/Llama-3.2-1B-Instruct,llama3_2_1b,mmlu,1,5,"Let A be a finite set with m elements, and let...","Let A be a finite set with m elements, and let...",n^m || n!/(n - m)! || n! || n!/(m!(n - m)!),NaN,NaN,0.0,C,False,C,1,67.9756,122.0342,18.4985,5.5667,7.3,9364.48,895.34,68.6667,68.750,3.0,3.0,5254.0,54.7267,55.12,llama3_2_1b,2026-04-07_13-35-14,mmlu,/home/stephane/resource-efficiency-benchmark/r...,NaN


In [13]:
list(samples.columns)


['repo_id',
 'served_model_id',
 'model_alias',
 'dataset_name',
 'run_idx',
 'sample_idx',
 'prompt',
 'question',
 'choices',
 'instruction',
 'context',
 'gold_answer',
 'parsed_prediction',
 'correct',
 'raw_output',
 'output_tokens',
 'ttft_ms',
 'end_to_end_latency_ms',
 'tokens_per_sec',
 'cpu_percent_avg',
 'cpu_percent_peak',
 'ram_used_mb_peak',
 'process_ram_mb_peak',
 'temp_c_avg',
 'temp_c_peak',
 'gpu_util_percent_avg',
 'gpu_util_percent_peak',
 'vram_used_mb_peak',
 'power_w_avg',
 'power_w_peak',
 'model_alias_dir',
 'run_id',
 'dataset',
 'source_path',
 'metric_points_json']

In [14]:
if not samples.empty:
    display(
        samples.groupby(["model_alias_dir", "dataset"])
        .size()
        .rename("n_rows")
        .reset_index()
        .sort_values(["model_alias_dir", "dataset"])
    )


,model_alias_dir,dataset,n_rows
0,llama3_2_1b,dolly,100
1,llama3_2_1b,gsm8k,1000
2,llama3_2_1b,hellaswag,1000
3,llama3_2_1b,mmlu,1000
4,ministral3_3b_awq,dolly,100
5,ministral3_3b_awq,gsm8k,1000
6,ministral3_3b_awq,hellaswag,1000
7,ministral3_3b_awq,mmlu,1000
8,phi4mini_gptq_4bit,dolly,100
9,phi4mini_gptq_4bit,gsm8k,1000


 vLLM summary schema checks

In [15]:
if not summaries.empty:
    summary_schema = (
        summaries.groupby("dataset")
        .apply(lambda x: pd.Series({
            "n_rows": len(x),
            "n_unique_source_files": x["source_path"].nunique(),
            "columns_present": list(x.columns),
        }))
        .reset_index()
    )
    display(summary_schema)


,dataset,n_rows,n_unique_source_files,columns_present
0,dolly,5,5,"[Model alias, Repo id, Served model id, Accura..."
1,gsm8k,5,5,"[Model alias, Repo id, Served model id, Accura..."
2,hellaswag,5,5,"[Model alias, Repo id, Served model id, Accura..."
3,mmlu,5,5,"[Model alias, Repo id, Served model id, Accura..."


In [16]:
if not summaries.empty:
    summary_row_counts = (
        summaries.groupby(["model_alias_dir", "run_id", "dataset"])
        .size()
        .rename("rows_per_summary_file")
        .reset_index()
        .sort_values(["model_alias_dir", "run_id", "dataset"])
    )
    display(summary_row_counts)


,model_alias_dir,run_id,dataset,rows_per_summary_file
0,llama3_2_1b,2026-04-07_13-35-14,dolly,1
1,llama3_2_1b,2026-04-07_13-35-14,gsm8k,1
2,llama3_2_1b,2026-04-07_13-35-14,hellaswag,1
3,llama3_2_1b,2026-04-07_13-35-14,mmlu,1
4,ministral3_3b_awq,2026-04-07_13-56-00,dolly,1
5,ministral3_3b_awq,2026-04-07_13-56-00,gsm8k,1
6,ministral3_3b_awq,2026-04-07_13-56-00,hellaswag,1
7,ministral3_3b_awq,2026-04-07_13-56-00,mmlu,1
8,phi4mini_gptq_4bit,2026-04-07_16-29-17,dolly,1
9,phi4mini_gptq_4bit,2026-04-07_16-29-17,gsm8k,1


In [17]:
if not summaries.empty:
    bad_summary_files = summary_row_counts[summary_row_counts["rows_per_summary_file"] != 1]
    display(bad_summary_files)


,model_alias_dir,run_id,dataset,rows_per_summary_file


 Summary null checks

In [18]:
if not summaries.empty:
    display(summaries.isna().sum().sort_values(ascending=False).head(40))


Accuracy sd                              5
Accuracy mean                            5
Repo id                                  0
Model alias                              0
Served model id                          0
TTFT ms mean                             0
TTFT ms sd                               0
End-to-end latency ms mean               0
End-to-end latency ms sd                 0
Tokens/sec mean                          0
Tokens/sec sd                            0
CPU percent avg mean                     0
CPU percent avg sd                       0
Temp C avg mean                          0
Temp C avg sd                            0
GPU percent avg mean                     0
GPU percent avg sd                       0
Power W avg mean                         0
Power W avg sd                           0
Sequential sustained tokens/sec mean     0
Sequential sustained tokens/sec sd       0
Sequential sustained samples/sec mean    0
Sequential sustained samples/sec sd      0
CPU percent

 vLLM-specific summary columns to verify

In [19]:
expected_vllm_summary_cols = [
    "Model alias",
    "Repo id",
    "Served model id",
    "TTFT ms mean",
    "End-to-end latency ms mean",
    "Tokens/sec mean",
    "CPU percent avg mean",
    "Process RAM MB peak",
    "GPU percent avg mean",
    "GPU percent peak",
    "VRAM used MB peak",
    "Power W avg mean",
    "Power W peak",
]

present_summary_cols = set(summaries.columns)
summary_col_check = pd.DataFrame({
    "column": expected_vllm_summary_cols,
    "present": [c in present_summary_cols for c in expected_vllm_summary_cols],
})
summary_col_check


,column,present
0,Model alias,True
1,Repo id,True
2,Served model id,True
3,TTFT ms mean,True
4,End-to-end latency ms mean,True
5,Tokens/sec mean,True
6,CPU percent avg mean,True
7,Process RAM MB peak,True
8,GPU percent avg mean,True
9,GPU percent peak,True


 Numeric sanity checks on vLLM summary metrics

In [20]:
summary_numeric_cols = [
    "Accuracy mean",
    "TTFT ms mean",
    "End-to-end latency ms mean",
    "Tokens/sec mean",
    "CPU percent avg mean",
    "CPU percent peak",
    "RAM used MB peak",
    "Process RAM MB peak",
    "Temp C avg mean",
    "Temp C peak",
    "GPU percent avg mean",
    "GPU percent peak",
    "VRAM used MB peak",
    "Power W avg mean",
    "Power W peak",
    "Sequential sustained tokens/sec mean",
    "Sequential sustained samples/sec mean",
]

summary_numeric_report = []

for col in summary_numeric_cols:
    if col in summaries.columns:
        vals = pd.to_numeric(summaries[col], errors="coerce")
        summary_numeric_report.append({
            "column": col,
            "min": vals.min(),
            "max": vals.max(),
            "n_negative": int((vals < 0).sum()),
            "n_null": int(vals.isna().sum()),
        })

pd.DataFrame(summary_numeric_report)


,column,min,max,n_negative,n_null
0,Accuracy mean,0.0300,0.7030,0,5
1,TTFT ms mean,67.0925,111.1288,0,0
2,End-to-end latency ms mean,122.4676,634.2238,0,0
3,Tokens/sec mean,14.6875,109.4824,0,0
4,CPU percent avg mean,5.5114,10.1223,0,0
5,CPU percent peak,14.4000,43.4000,0,0
6,RAM used MB peak,6796.8800,10783.7600,0,0
7,Process RAM MB peak,820.4500,1149.6000,0,0
8,Temp C avg mean,67.6965,77.8582,0,0
9,Temp C peak,68.6250,79.8750,0,0


 vLLM sample schema checks

In [21]:
if not samples.empty:
    sample_row_counts = (
        samples.groupby(["model_alias_dir", "run_id", "dataset"])
        .size()
        .rename("rows_per_samples_file")
        .reset_index()
        .sort_values(["model_alias_dir", "run_id", "dataset"])
    )
    display(sample_row_counts)


,model_alias_dir,run_id,dataset,rows_per_samples_file
0,llama3_2_1b,2026-04-07_13-35-14,dolly,100
1,llama3_2_1b,2026-04-07_13-35-14,gsm8k,1000
2,llama3_2_1b,2026-04-07_13-35-14,hellaswag,1000
3,llama3_2_1b,2026-04-07_13-35-14,mmlu,1000
4,ministral3_3b_awq,2026-04-07_13-56-00,dolly,100
5,ministral3_3b_awq,2026-04-07_13-56-00,gsm8k,1000
6,ministral3_3b_awq,2026-04-07_13-56-00,hellaswag,1000
7,ministral3_3b_awq,2026-04-07_13-56-00,mmlu,1000
8,phi4mini_gptq_4bit,2026-04-07_16-29-17,dolly,100
9,phi4mini_gptq_4bit,2026-04-07_16-29-17,gsm8k,1000


In [22]:
expected_vllm_sample_cols = [
    "repo_id",
    "served_model_id",
    "model_alias",
    "dataset_name",
    "run_idx",
    "sample_idx",
    "output_tokens",
    "ttft_ms",
    "end_to_end_latency_ms",
    "tokens_per_sec",
    "cpu_percent_avg",
    "cpu_percent_peak",
    "process_ram_mb_peak",
    "gpu_util_percent_avg",
    "gpu_util_percent_peak",
    "vram_used_mb_peak",
    "power_w_avg",
    "power_w_peak",
]

present_sample_cols = set(samples.columns)
sample_col_check = pd.DataFrame({
    "column": expected_vllm_sample_cols,
    "present": [c in present_sample_cols for c in expected_vllm_sample_cols],
})
sample_col_check


,column,present
0,repo_id,True
1,served_model_id,True
2,model_alias,True
3,dataset_name,True
4,run_idx,True
5,sample_idx,True
6,output_tokens,True
7,ttft_ms,True
8,end_to_end_latency_ms,True
9,tokens_per_sec,True


 Per-run sample count consistency (vLLM)

In [23]:
if not samples.empty and "run_idx" in samples.columns:
    per_run_counts = (
        samples.groupby(["model_alias_dir", "run_id", "dataset", "run_idx"])
        .size()
        .rename("n_samples")
        .reset_index()
        .sort_values(["model_alias_dir", "run_id", "dataset", "run_idx"])
    )
    display(per_run_counts.head(100))


,model_alias_dir,run_id,dataset,run_idx,n_samples
0,llama3_2_1b,2026-04-07_13-35-14,dolly,1,10
1,llama3_2_1b,2026-04-07_13-35-14,dolly,2,10
2,llama3_2_1b,2026-04-07_13-35-14,dolly,3,10
3,llama3_2_1b,2026-04-07_13-35-14,dolly,4,10
4,llama3_2_1b,2026-04-07_13-35-14,dolly,5,10
5,llama3_2_1b,2026-04-07_13-35-14,dolly,6,10
6,llama3_2_1b,2026-04-07_13-35-14,dolly,7,10
7,llama3_2_1b,2026-04-07_13-35-14,dolly,8,10
8,llama3_2_1b,2026-04-07_13-35-14,dolly,9,10
9,llama3_2_1b,2026-04-07_13-35-14,dolly,10,10


In [24]:
if not samples.empty and "run_idx" in samples.columns:
    run_consistency = (
        per_run_counts.groupby(["model_alias_dir", "run_id", "dataset"])["n_samples"]
        .agg(["min", "max", "nunique"])
        .reset_index()
        .sort_values(["model_alias_dir", "run_id", "dataset"])
    )
    display(run_consistency)


,model_alias_dir,run_id,dataset,min,max,nunique
0,llama3_2_1b,2026-04-07_13-35-14,dolly,10,10,1
1,llama3_2_1b,2026-04-07_13-35-14,gsm8k,100,100,1
2,llama3_2_1b,2026-04-07_13-35-14,hellaswag,100,100,1
3,llama3_2_1b,2026-04-07_13-35-14,mmlu,100,100,1
4,ministral3_3b_awq,2026-04-07_13-56-00,dolly,10,10,1
5,ministral3_3b_awq,2026-04-07_13-56-00,gsm8k,100,100,1
6,ministral3_3b_awq,2026-04-07_13-56-00,hellaswag,100,100,1
7,ministral3_3b_awq,2026-04-07_13-56-00,mmlu,100,100,1
8,phi4mini_gptq_4bit,2026-04-07_16-29-17,dolly,10,10,1
9,phi4mini_gptq_4bit,2026-04-07_16-29-17,gsm8k,100,100,1


 Check that warmup was excluded from saved vLLM sample files

In [25]:
if not samples.empty and "run_idx" in samples.columns:
    warmup_check = (
        samples.groupby("dataset")["run_idx"]
        .agg(["min", "max", "nunique"])
        .reset_index()
    )
    display(warmup_check)


,dataset,min,max,nunique
0,dolly,1,10,10
1,gsm8k,1,10,10
2,hellaswag,1,10,10
3,mmlu,1,10,10


 Numeric sanity checks on vLLM sample metrics

In [26]:
sample_numeric_cols = [
    "output_tokens",
    "ttft_ms",
    "end_to_end_latency_ms",
    "tokens_per_sec",
    "cpu_percent_avg",
    "cpu_percent_peak",
    "ram_used_mb_peak",
    "process_ram_mb_peak",
    "temp_c_avg",
    "temp_c_peak",
    "gpu_util_percent_avg",
    "gpu_util_percent_peak",
    "vram_used_mb_peak",
    "power_w_avg",
    "power_w_peak",
]

sample_numeric_report = []

for col in sample_numeric_cols:
    if col in samples.columns:
        vals = pd.to_numeric(samples[col], errors="coerce")
        sample_numeric_report.append({
            "column": col,
            "min": vals.min(),
            "max": vals.max(),
            "n_negative": int((vals < 0).sum()),
            "n_null": int(vals.isna().sum()),
        })

pd.DataFrame(sample_numeric_report)


,column,min,max,n_negative,n_null
0,output_tokens,1.0000,68.0000,0,0
1,ttft_ms,64.0132,308.3794,0,0
2,end_to_end_latency_ms,117.6856,990.9335,0,0
3,tokens_per_sec,8.7443,151.9511,0,0
4,cpu_percent_avg,3.1333,26.2000,0,0
5,cpu_percent_peak,4.9000,43.4000,0,0
6,ram_used_mb_peak,6681.2300,10783.7600,0,0
7,process_ram_mb_peak,818.8000,1149.6000,0,0
8,temp_c_avg,48.5833,79.6667,0,0
9,temp_c_peak,49.2500,79.8750,0,0


 vLLM GPU / VRAM / power focused checks

In [27]:
gpu_focus_cols = [
    "gpu_util_percent_avg",
    "gpu_util_percent_peak",
    "vram_used_mb_peak",
    "power_w_avg",
    "power_w_peak",
]

gpu_focus_report = []

for col in gpu_focus_cols:
    if col in samples.columns:
        vals = pd.to_numeric(samples[col], errors="coerce")
        gpu_focus_report.append({
            "column": col,
            "min": vals.min(),
            "p50": vals.median(),
            "max": vals.max(),
            "null_rate": vals.isna().mean(),
        })

pd.DataFrame(gpu_focus_report)


,column,min,p50,max,null_rate
0,gpu_util_percent_avg,0.00,20.00,100.0000,0.0
1,gpu_util_percent_peak,0.00,21.00,100.0000,0.0
2,vram_used_mb_peak,5145.00,5370.00,5941.0000,0.0
3,power_w_avg,21.20,71.07,191.8125,0.0
4,power_w_peak,22.47,72.11,196.3300,0.0


 Accuracy / correctness inspection for vLLM runs

In [28]:
if "correct" in samples.columns:
    correctness_by_dataset = (
        samples.groupby("dataset")["correct"]
        .agg(["count", "mean"])
        .reset_index()
    )
    display(correctness_by_dataset)


,dataset,count,mean
0,dolly,0,NaN
1,gsm8k,5000,0.076
2,hellaswag,5000,0.4302
3,mmlu,5000,0.5686


In [29]:
if "parsed_prediction" in samples.columns:
    parsed_prediction_coverage = (
        samples.groupby("dataset")["parsed_prediction"]
        .apply(lambda s: s.notna().mean())
        .rename("parsed_prediction_non_null_rate")
        .reset_index()
    )
    display(parsed_prediction_coverage)


,dataset,parsed_prediction_non_null_rate
0,dolly,0.0000
1,gsm8k,0.8984
2,hellaswag,1.0000
3,mmlu,0.9920


 vLLM identity fields inspection

In [30]:
identity_cols = [c for c in ["Model alias", "Repo id", "Served model id"] if c in summaries.columns]
if identity_cols:
    display(summaries[identity_cols + ["model_alias_dir", "run_id", "dataset"]].head(20))


,Model alias,Repo id,Served model id,model_alias_dir,run_id,dataset
0,llama3_2_1b,unsloth/Llama-3.2-1B-Instruct,unsloth/Llama-3.2-1B-Instruct,llama3_2_1b,2026-04-07_13-35-14,mmlu
1,llama3_2_1b,unsloth/Llama-3.2-1B-Instruct,unsloth/Llama-3.2-1B-Instruct,llama3_2_1b,2026-04-07_13-35-14,hellaswag
2,llama3_2_1b,unsloth/Llama-3.2-1B-Instruct,unsloth/Llama-3.2-1B-Instruct,llama3_2_1b,2026-04-07_13-35-14,gsm8k
3,llama3_2_1b,unsloth/Llama-3.2-1B-Instruct,unsloth/Llama-3.2-1B-Instruct,llama3_2_1b,2026-04-07_13-35-14,dolly
4,ministral3_3b_awq,cyankiwi/Ministral-3-3B-Instruct-2512-AWQ-4bit,cyankiwi/Ministral-3-3B-Instruct-2512-AWQ-4bit,ministral3_3b_awq,2026-04-07_13-56-00,mmlu
5,ministral3_3b_awq,cyankiwi/Ministral-3-3B-Instruct-2512-AWQ-4bit,cyankiwi/Ministral-3-3B-Instruct-2512-AWQ-4bit,ministral3_3b_awq,2026-04-07_13-56-00,hellaswag
6,ministral3_3b_awq,cyankiwi/Ministral-3-3B-Instruct-2512-AWQ-4bit,cyankiwi/Ministral-3-3B-Instruct-2512-AWQ-4bit,ministral3_3b_awq,2026-04-07_13-56-00,gsm8k
7,ministral3_3b_awq,cyankiwi/Ministral-3-3B-Instruct-2512-AWQ-4bit,cyankiwi/Ministral-3-3B-Instruct-2512-AWQ-4bit,ministral3_3b_awq,2026-04-07_13-56-00,dolly
8,phi4mini_gptq_4bit,iqbalamo93/Phi-4-mini-instruct-GPTQ-4bit,iqbalamo93/Phi-4-mini-instruct-GPTQ-4bit,phi4mini_gptq_4bit,2026-04-07_16-29-17,mmlu
9,phi4mini_gptq_4bit,iqbalamo93/Phi-4-mini-instruct-GPTQ-4bit,iqbalamo93/Phi-4-mini-instruct-GPTQ-4bit,phi4mini_gptq_4bit,2026-04-07_16-29-17,hellaswag


In [31]:
identity_cols_samples = [c for c in ["model_alias", "repo_id", "served_model_id"] if c in samples.columns]
if identity_cols_samples:
    display(samples[identity_cols_samples + ["model_alias_dir", "run_id", "dataset"]].head(20))


,model_alias,repo_id,served_model_id,model_alias_dir,run_id,dataset
0,llama3_2_1b,unsloth/Llama-3.2-1B-Instruct,unsloth/Llama-3.2-1B-Instruct,llama3_2_1b,2026-04-07_13-35-14,mmlu
1,llama3_2_1b,unsloth/Llama-3.2-1B-Instruct,unsloth/Llama-3.2-1B-Instruct,llama3_2_1b,2026-04-07_13-35-14,mmlu
2,llama3_2_1b,unsloth/Llama-3.2-1B-Instruct,unsloth/Llama-3.2-1B-Instruct,llama3_2_1b,2026-04-07_13-35-14,mmlu
3,llama3_2_1b,unsloth/Llama-3.2-1B-Instruct,unsloth/Llama-3.2-1B-Instruct,llama3_2_1b,2026-04-07_13-35-14,mmlu
4,llama3_2_1b,unsloth/Llama-3.2-1B-Instruct,unsloth/Llama-3.2-1B-Instruct,llama3_2_1b,2026-04-07_13-35-14,mmlu
5,llama3_2_1b,unsloth/Llama-3.2-1B-Instruct,unsloth/Llama-3.2-1B-Instruct,llama3_2_1b,2026-04-07_13-35-14,mmlu
6,llama3_2_1b,unsloth/Llama-3.2-1B-Instruct,unsloth/Llama-3.2-1B-Instruct,llama3_2_1b,2026-04-07_13-35-14,mmlu
7,llama3_2_1b,unsloth/Llama-3.2-1B-Instruct,unsloth/Llama-3.2-1B-Instruct,llama3_2_1b,2026-04-07_13-35-14,mmlu
8,llama3_2_1b,unsloth/Llama-3.2-1B-Instruct,unsloth/Llama-3.2-1B-Instruct,llama3_2_1b,2026-04-07_13-35-14,mmlu
9,llama3_2_1b,unsloth/Llama-3.2-1B-Instruct,unsloth/Llama-3.2-1B-Instruct,llama3_2_1b,2026-04-07_13-35-14,mmlu


 Dolly special case in vLLM outputs

In [32]:
dolly_summary = summaries[summaries["dataset"] == "dolly"].copy()
print("vLLM dolly summary shape:", dolly_summary.shape)
display(dolly_summary.head())


vLLM dolly summary shape: (5, 34)


,Model alias,Repo id,Served model id,Accuracy mean,Accuracy sd,TTFT ms mean,TTFT ms sd,End-to-end latency ms mean,End-to-end latency ms sd,Tokens/sec mean,Tokens/sec sd,CPU percent avg mean,CPU percent avg sd,Temp C avg mean,Temp C avg sd,GPU percent avg mean,GPU percent avg sd,Power W avg mean,Power W avg sd,Sequential sustained tokens/sec mean,Sequential sustained tokens/sec sd,Sequential sustained samples/sec mean,Sequential sustained samples/sec sd,CPU percent peak,RAM used MB peak,Process RAM MB peak,Temp C peak,GPU percent peak,VRAM used MB peak,Power W peak,model_alias_dir,run_id,dataset,source_path
3,llama3_2_1b,unsloth/Llama-3.2-1B-Instruct,unsloth/Llama-3.2-1B-Instruct,NaN,NaN,73.7217,5.8875,471.5200,54.9763,109.4824,4.3976,7.9899,0.2048,74.9008,0.1878,66.0237,30.1451,158.7051,4.3896,83.8610,4.9172,1.9239,0.1934,22.2,9301.80,899.69,76.250,100.0,5158.0,187.04,llama3_2_1b,2026-04-07_13-35-14,dolly,/home/stephane/resource-efficiency-benchmark/r...
7,ministral3_3b_awq,cyankiwi/Ministral-3-3B-Instruct-2512-AWQ-4bit,cyankiwi/Ministral-3-3B-Instruct-2512-AWQ-4bit,NaN,NaN,86.5122,16.6781,582.3180,34.5942,86.0036,3.6222,8.0497,0.1807,75.5954,0.2432,34.0720,35.4305,170.7876,3.6607,67.4267,3.6057,1.5745,0.0923,22.2,10620.04,909.82,77.125,100.0,5165.0,195.87,ministral3_3b_awq,2026-04-07_13-56-00,dolly,/home/stephane/resource-efficiency-benchmark/r...
11,phi4mini_gptq_4bit,iqbalamo93/Phi-4-mini-instruct-GPTQ-4bit,iqbalamo93/Phi-4-mini-instruct-GPTQ-4bit,NaN,NaN,92.8500,17.0872,634.2238,53.6349,86.4342,6.7761,7.3774,0.1632,75.3664,0.3586,75.1083,24.7894,169.1363,4.1583,67.7461,5.6492,1.4310,0.1098,14.4,6796.88,881.47,77.125,100.0,5930.0,196.33,phi4mini_gptq_4bit,2026-04-07_16-29-17,dolly,/home/stephane/resource-efficiency-benchmark/r...
15,qwen2_5_1_5b,Qwen/Qwen2.5-1.5B-Instruct,Qwen/Qwen2.5-1.5B-Instruct,NaN,NaN,87.3424,8.1712,612.8584,62.1703,77.1278,5.1576,10.1223,0.2733,77.8582,0.4597,56.1125,33.1230,154.1003,4.9660,61.9599,5.8608,1.4967,0.1363,23.3,9716.25,823.19,79.875,100.0,5404.0,180.03,qwen2_5_1_5b,2026-04-07_14-29-28,dolly,/home/stephane/resource-efficiency-benchmark/r...
19,qwen3_4b_awq,cyankiwi/Qwen3-4B-Instruct-2507-AWQ-4bit,cyankiwi/Qwen3-4B-Instruct-2507-AWQ-4bit,NaN,NaN,100.5900,22.6301,632.8397,51.0374,85.4784,4.4058,7.4515,0.1932,75.8978,0.2042,62.6153,41.9713,170.2425,3.3633,65.8724,5.1170,1.4282,0.1123,18.7,9781.43,849.93,77.375,100.0,5254.0,195.27,qwen3_4b_awq,2026-04-07_14-50-20,dolly,/home/stephane/resource-efficiency-benchmark/r...


In [33]:
dolly_samples = samples[samples["dataset"] == "dolly"].copy()
print("vLLM dolly samples shape:", dolly_samples.shape)
display(dolly_samples.head())


vLLM dolly samples shape: (500, 35)


,repo_id,served_model_id,model_alias,dataset_name,run_idx,sample_idx,prompt,question,choices,instruction,context,gold_answer,parsed_prediction,correct,raw_output,output_tokens,ttft_ms,end_to_end_latency_ms,tokens_per_sec,cpu_percent_avg,cpu_percent_peak,ram_used_mb_peak,process_ram_mb_peak,temp_c_avg,temp_c_peak,gpu_util_percent_avg,gpu_util_percent_peak,vram_used_mb_peak,power_w_avg,power_w_peak,model_alias_dir,run_id,dataset,source_path,metric_points_json
3000,unsloth/Llama-3.2-1B-Instruct,unsloth/Llama-3.2-1B-Instruct,llama3_2_1b,dolly,1,1,Identify which instrument is string or percuss...,NaN,NaN,Identify which instrument is string or percuss...,NaN,NaN,NaN,NaN,"Pipa is a string instrument, while Tifa is a p...",11,67.9216,167.0385,110.9801,6.1667,9.6,9203.11,899.62,72.7083,72.750,0.0,0.0,5158.0,69.1333,76.70,llama3_2_1b,2026-04-07_13-35-14,dolly,/home/stephane/resource-efficiency-benchmark/r...,NaN
3001,unsloth/Llama-3.2-1B-Instruct,unsloth/Llama-3.2-1B-Instruct,llama3_2_1b,dolly,1,2,Give me a list of the Mercury Prize winning al...,NaN,NaN,Give me a list of the Mercury Prize winning al...,NaN,NaN,NaN,NaN,Here's a list of Mercury Prize winning albums ...,38,66.9484,557.2435,77.5043,8.2600,9.5,9193.40,899.62,73.3250,73.875,0.0,0.0,5158.0,104.7120,137.58,llama3_2_1b,2026-04-07_13-35-14,dolly,/home/stephane/resource-efficiency-benchmark/r...,NaN
3002,unsloth/Llama-3.2-1B-Instruct,unsloth/Llama-3.2-1B-Instruct,llama3_2_1b,dolly,1,3,What should I do on a free afternoon in San Fr...,NaN,NaN,What should I do on a free afternoon in San Fr...,NaN,NaN,NaN,NaN,San Francisco is a vibrant city with endless o...,61,68.3661,558.5294,124.4483,8.5200,10.0,9184.02,899.62,74.1500,74.500,60.0,100.0,5158.0,159.4080,183.13,llama3_2_1b,2026-04-07_13-35-14,dolly,/home/stephane/resource-efficiency-benchmark/r...,NaN
3003,unsloth/Llama-3.2-1B-Instruct,unsloth/Llama-3.2-1B-Instruct,llama3_2_1b,dolly,1,4,Give me a list of actresses who won an Oscar i...,NaN,NaN,Give me a list of actresses who won an Oscar i...,NaN,NaN,NaN,NaN,Here's a list of actresses who won an Oscar in...,41,67.9221,558.2024,83.6256,8.3000,10.2,9203.76,899.62,74.7250,75.000,100.0,100.0,5158.0,176.5120,184.36,llama3_2_1b,2026-04-07_13-35-14,dolly,/home/stephane/resource-efficiency-benchmark/r...,NaN
3004,unsloth/Llama-3.2-1B-Instruct,unsloth/Llama-3.2-1B-Instruct,llama3_2_1b,dolly,1,5,Give me a list of the ways you can be 'out' in...,NaN,NaN,Give me a list of the ways you can be 'out' in...,NaN,NaN,NaN,NaN,"In cricket, there are several ways a player ca...",54,67.6954,558.0891,110.1156,8.0200,10.0,9212.45,899.62,75.1250,75.375,100.0,100.0,5158.0,174.4360,183.64,llama3_2_1b,2026-04-07_13-35-14,dolly,/home/stephane/resource-efficiency-benchmark/r...,NaN


 Final quick vLLM exploration summary

In [34]:
if not summaries.empty:
    print("vLLM model aliases discovered:", sorted(summaries["model_alias_dir"].dropna().unique()))
    print("vLLM datasets discovered:", sorted(summaries["dataset"].dropna().unique()))
    print("vLLM run_ids discovered:", summaries["run_id"].nunique())

if not samples.empty:
    sample_summary = (
        samples.groupby(["model_alias_dir", "dataset"])
        .size()
        .rename("n_sample_rows")
        .reset_index()
        .sort_values(["model_alias_dir", "dataset"])
    )
    display(sample_summary)

vLLM model aliases discovered: ['llama3_2_1b', 'ministral3_3b_awq', 'phi4mini_gptq_4bit', 'qwen2_5_1_5b', 'qwen3_4b_awq']
vLLM datasets discovered: ['dolly', 'gsm8k', 'hellaswag', 'mmlu']
vLLM run_ids discovered: 5


,model_alias_dir,dataset,n_sample_rows
0,llama3_2_1b,dolly,100
1,llama3_2_1b,gsm8k,1000
2,llama3_2_1b,hellaswag,1000
3,llama3_2_1b,mmlu,1000
4,ministral3_3b_awq,dolly,100
5,ministral3_3b_awq,gsm8k,1000
6,ministral3_3b_awq,hellaswag,1000
7,ministral3_3b_awq,mmlu,1000
8,phi4mini_gptq_4bit,dolly,100
9,phi4mini_gptq_4bit,gsm8k,1000
